# Load and Process Collected Dataset

This notebook processes both Kaggle and collected datasets for training.

In [1]:
import os
from os.path import dirname

root_dir = dirname(os.getcwd())
os.chdir(root_dir)

In [2]:
import torch
import pandas as pd

from src.utils import *

## Load Collected Dataset

In [3]:
def load_collected_data(folder_path):
    """Load accelerometer data from collected folder and convert to kaggle format."""
    accel_path = os.path.join(folder_path, 'Accelerometer.csv')
    df = pd.read_csv(accel_path)

    # Convert to kaggle format: time, ax, ay, az
    processed_df = pd.DataFrame({
        'time': df['timeIntervalSince1970'],
        'ax': df['x'],
        'ay': df['y'],
        'az': df['z']
    })

    return processed_df

In [4]:
# Load collected bike data
collected_bike_df = load_collected_data('data/collected/bike')

# Load all collected car data
collected_car_dfs = []
for i in range(1, 2):
    car_folder = f'data/collected/car-{i}'
    car_df = load_collected_data(car_folder)
    collected_car_dfs.append(car_df)
    print(f"Loaded car-{i}: {len(car_df)} rows")

# Concatenate all collected car data
collected_car_df = pd.concat(collected_car_dfs, ignore_index=True)

print(f"\nLoaded collected bike: {len(collected_bike_df)} rows")
print(f"Total collected car data: {len(collected_car_df)} rows")

Loaded car-1: 189334 rows

Loaded collected bike: 2628 rows
Total collected car data: 189334 rows


In [5]:
# Split collected data: first 90% train, last 10% test
split_ratio = 0.90

collected_bike_split_idx = int(len(collected_bike_df) * split_ratio)
collected_bike_train = collected_bike_df.iloc[:collected_bike_split_idx].reset_index(drop=True)
collected_bike_test = collected_bike_df.iloc[collected_bike_split_idx:].reset_index(drop=True)

collected_car_split_idx = int(len(collected_car_df) * split_ratio)
collected_car_train = collected_car_df.iloc[:collected_car_split_idx].reset_index(drop=True)
collected_car_test = collected_car_df.iloc[collected_car_split_idx:].reset_index(drop=True)

print("Collected Train set (first 90%):")
print(f"  Bike: {len(collected_bike_train)} rows")
print(f"  Car: {len(collected_car_train)} rows")

print("\nCollected Test set (last 10%):")
print(f"  Bike: {len(collected_bike_test)} rows")
print(f"  Car: {len(collected_car_test)} rows")

Collected Train set (first 90%):
  Bike: 2365 rows
  Car: 170400 rows

Collected Test set (last 10%):
  Bike: 263 rows
  Car: 18934 rows


In [6]:
# Process and save collected train dataset
collected_train_dataset = process_vehicle_dataset(
    [collected_bike_train, collected_car_train],
    ['bike', 'car'],
    'data/vehicle_data_collected_train.pkl',
)

Processed bike: 1423 windows with label value 0
Processed car: 2311 windows with label value 1

Dataset saved to data/vehicle_data_collected_train.pkl
Total samples: 3734
Data shape: torch.Size([3734, 3, 50])
Label mapping: {'bike': 0, 'car': 1}


In [7]:
# Process and save collected test dataset
collected_test_dataset = process_vehicle_dataset(
    [collected_bike_test, collected_car_test],
    ['bike', 'car'],
    'data/vehicle_data_collected_test.pkl',
)

Processed bike: 7 windows with label value 0
Processed car: 1559 windows with label value 1

Dataset saved to data/vehicle_data_collected_test.pkl
Total samples: 1566
Data shape: torch.Size([1566, 3, 50])
Label mapping: {'bike': 0, 'car': 1}


In [8]:
# Compare collected dataset statistics
print("\n" + "="*60)
print("COLLECTED DATASET COMPARISON")
print("="*60)

collected_train_total = collected_train_dataset['data'].shape[0]
collected_test_total = collected_test_dataset['data'].shape[0]

print(f"\nTotal windows in collected train set: {collected_train_total}")
print(f"Total windows in collected test set: {collected_test_total}")
print(f"Ratio (test/train): {collected_test_total/collected_train_total:.2%}")

print(f"\nCollected train label distribution: {torch.bincount(collected_train_dataset['label'])}")
print(f"Collected test label distribution: {torch.bincount(collected_test_dataset['label'])}")


COLLECTED DATASET COMPARISON

Total windows in collected train set: 3734
Total windows in collected test set: 1566
Ratio (test/train): 41.94%

Collected train label distribution: tensor([1423, 2311])
Collected test label distribution: tensor([   7, 1559])


## Load Kaggle Dataset

In [9]:
# Load raw data
bike_df = pd.read_csv('data/kaggle/bike.csv')

# Load all car data files
car_clear_df = pd.read_csv('data/kaggle/car-clear.csv')
# car_clear2_df = pd.read_csv('data/kaggle/car-clear2.csv')
# car_rain_df = pd.read_csv('data/kaggle/car-rain.csv')
# car_city_df = pd.read_csv('data/kaggle/car-city-newark-light-rain.csv')

# Concatenate all car data
car_df = car_clear_df

print(f"Loaded bike: {len(bike_df)} rows")
print(f"Loaded car-clear: {len(car_clear_df)} rows")
print(f"Total car data: {len(car_df)} rows")

Loaded bike: 35515 rows
Loaded car-clear: 44416 rows
Total car data: 44416 rows


In [10]:
# Slice each time series: first 90% train, last 10% test
split_ratio = 0.90

# Bike split
bike_split_idx = int(len(bike_df) * split_ratio)
bike_train = bike_df.iloc[:bike_split_idx].reset_index(drop=True)
bike_test = bike_df.iloc[bike_split_idx:].reset_index(drop=True)

# Car split
car_split_idx = int(len(car_df) * split_ratio)
car_train = car_df.iloc[:car_split_idx].reset_index(drop=True)
car_test = car_df.iloc[car_split_idx:].reset_index(drop=True)

print("Train set (first 90%):")
print(f"  Bike: {len(bike_train)} rows")
print(f"  Car: {len(car_train)} rows")

print("\nTest set (last 10%):")
print(f"  Bike: {len(bike_test)} rows")
print(f"  Car: {len(car_test)} rows")

Train set (first 90%):
  Bike: 31963 rows
  Car: 39974 rows

Test set (last 10%):
  Bike: 3552 rows
  Car: 4442 rows


In [11]:
train_dataset = process_vehicle_dataset(
    [bike_train, car_train],
    ['bike', 'car'],
    'data/vehicle_data_kaggle_train.pkl',
)

Processed bike: 269 windows with label value 0
Processed car: 355 windows with label value 1

Dataset saved to data/vehicle_data_kaggle_train.pkl
Total samples: 624
Data shape: torch.Size([624, 3, 50])
Label mapping: {'bike': 0, 'car': 1}


In [12]:
test_dataset = process_vehicle_dataset(
    [bike_test, car_test],
    ['bike', 'car'],
    'data/vehicle_data_kaggle_test.pkl',
)

Processed bike: 29 windows with label value 0
Processed car: 43 windows with label value 1

Dataset saved to data/vehicle_data_kaggle_test.pkl
Total samples: 72
Data shape: torch.Size([72, 3, 50])
Label mapping: {'bike': 0, 'car': 1}


In [13]:
# Compare final window counts
print("\n" + "="*60)
print("FINAL COMPARISON")
print("="*60)

train_total = train_dataset['data'].shape[0]
test_total = test_dataset['data'].shape[0]

print(f"\nTotal windows in train set: {train_total}")
print(f"Total windows in test set: {test_total}")
print(f"Ratio (test/train): {test_total/train_total:.2%}")

print(f"\nTrain label distribution: {torch.bincount(train_dataset['label'])}")
print(f"Test label distribution: {torch.bincount(test_dataset['label'])}")


FINAL COMPARISON

Total windows in train set: 624
Total windows in test set: 72
Ratio (test/train): 11.54%

Train label distribution: tensor([269, 355])
Test label distribution: tensor([29, 43])
